# G1 Academy Bonus - Task 6: basic locomotion helpers

## Introduction
Two locomotion helpers, both built on the native `LocoClient`: `loco_move` runs a bounded-duration continuous velocity command and always stops in `finally`; `odom_move` issues a single non-continuous relative move and closes the loop on odometry to confirm arrival, instead of treating the RPC return code alone as proof of motion. A simple lock prevents two locomotion commands from running concurrently.

In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelPublisher, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    if _factory_config is None:
        ChannelFactoryInitialize(*config)
        _factory_config = config
    elif _factory_config != config:
        raise RuntimeError(f"ChannelFactory already initialized as {_factory_config}; restart kernel for {config}.")
    return _factory_config

ensure_channel_factory(0, "eth0")

class Latest:
    def __init__(self, topic, message_type, queue_len=10):
        self.message = None
        self.timestamp = 0.0
        self.subscriber = ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self._callback, queue_len)
    def _callback(self, message):
        self.message = message
        self.timestamp = time.time()
    def fresh(self, max_age_s=0.5):
        return self.message is not None and time.time() - self.timestamp <= max_age_s

lowstate_sub = Latest("rt/lowstate", LowState_)

## Task 1 - Odometry subscriber + native `LocoClient`

In [ ]:
import math
import threading
from unitree_sdk2py.g1.loco.g1_loco_client import LocoClient
from unitree_sdk2py.idl.unitree_go.msg.dds_ import SportModeState_

loco = LocoClient(); loco.SetTimeout(5.0); loco.Init()
odom_sub = Latest("rt/odommodestate", SportModeState_)

def _odom_pose():
    msg = odom_sub.message
    if msg is None:
        return None
    return (float(msg.position[0]), float(msg.position[1]), float(msg.imu_state.rpy[2]))

def loco_stop():
    if hasattr(loco, "StopMove"):
        return loco.StopMove()
    return loco.Move(0.0, 0.0, 0.0, continous_move=False)

_locomotion_lock = threading.Lock()

## Task 2 - `loco_move(vx, vy, vyaw, duration_s)`
A continuous velocity command held for a fixed duration, always cancelled in `finally` - so an exception mid-sleep (kernel interrupt, timeout) can never leave the robot moving.

In [ ]:
def loco_move(vx, vy, vyaw, duration_s=2.0):
    if not _locomotion_lock.acquire(blocking=False):
        raise RuntimeError("Another locomotion command is already in progress.")
    try:
        code = int(loco.Move(float(vx), float(vy), float(vyaw), continous_move=True) or 0)
        try:
            time.sleep(max(0.0, float(duration_s)))
        finally:
            loco_stop()
        return code
    finally:
        _locomotion_lock.release()

# loco_move(0.03, 0.0, 0.0, duration_s=0.5)

## Task 3 - `odom_move(target_dx, target_dy, target_dyaw)` with closed-loop arrival monitoring
Issue one non-continuous `Move` request for the relative displacement, then poll odometry until the *observed* displacement is within tolerance, the request was rejected, or `timeout_s` elapses - stopping the robot and reporting failure on cancellation/timeout instead of hanging.

In [ ]:
def odom_move(target_dx, target_dy, target_dyaw, pos_tol_m=0.05, yaw_tol_rad=0.05, timeout_s=15.0, poll_s=0.2):
    if not _locomotion_lock.acquire(blocking=False):
        raise RuntimeError("Another locomotion command is already in progress.")
    try:
        start = _odom_pose()
        if start is None:
            raise RuntimeError("No fresh odometry; cannot monitor arrival.")
        code = int(loco.Move(float(target_dx), float(target_dy), float(target_dyaw), continous_move=False) or 0)
        if code != 0:
            return {"code": code, "arrived": False, "reason": "move request rejected"}
        deadline = time.time() + timeout_s
        pose = None
        while time.time() < deadline:
            pose = _odom_pose()
            if pose is not None:
                dx, dy, dyaw = pose[0] - start[0], pose[1] - start[1], pose[2] - start[2]
                err_pos = ((target_dx - dx) ** 2 + (target_dy - dy) ** 2) ** 0.5
                err_yaw = abs(((target_dyaw - dyaw) + math.pi) % (2 * math.pi) - math.pi)
                if err_pos <= pos_tol_m and err_yaw <= yaw_tol_rad:
                    return {"code": code, "arrived": True, "moved": (dx, dy, dyaw)}
            time.sleep(poll_s)
        loco_stop()
        moved = None if pose is None else (pose[0] - start[0], pose[1] - start[1], pose[2] - start[2])
        return {"code": code, "arrived": False, "reason": "timeout", "moved": moved}
    finally:
        _locomotion_lock.release()

# odom_move(0.2, 0.0, 0.0)

### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.